# NNMF analysis

Run the setup cell first. It opens one window to choose the working folder, reference workbook, data subfolders, and analysis parameters. Then run the remaining cells from top to bottom to inspect preprocessing, fit all original spectra jointly, export results, and create plots. Rerun the setup cell to configure a new analysis.

Each original spectrum has its baseline removed, is scaled at NV0, cropped, and normalized by its own trapezoidal area following the MATLAB workflow. Folder averages are arithmetic means for inspection/export only; NNMF learns shared shapes from all selected original spectra together. When reference initialization is enabled, one replicate starts from workbook references and the remaining replicates start randomly.

Results are saved under `export/`, with figures in `export/figures/` and folder averages in `export/folder_averages/`.


In [ ]:
from pathlib import Path
import importlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import nnmf_full_fixed_jh_V3 as nnmf
nnmf = importlib.reload(nnmf)

COMPONENT_NAMES = ["NV0", "NV-", "H3", "laser"]
DEFAULT_ANALYSIS_PARAMETERS = {
    "COMPONENTS": 2,
    "INIT_COMP": 2,
    "BASEMEAN_LOWER": 377,
    "BASEMEAN_UPPER": 480,
    "THRESHOLD_LOWER": 540,
    "THRESHOLD_UPPER": 900,
    "THRESHOLD_LOWER2": 540,
    "THRESHOLD_UPPER2": 900,
    "NNMF_REPLICATES": 20,
    "MAX_ITERATIONS": 100,
    "RANDOM_SEED": 21,
    "NNMF_NEGATIVE_TOLERANCE": 5e-4,
    "NV0": 575,
    "NVN": 640,
    "SHOW_COMPARED": True,
}

nnmf.configure_analysis({
    "DATA_FOLDER": Path.cwd(),
    "REFERENCE_FILE": None,
    "COMPONENT_NAMES": COMPONENT_NAMES,
    "EXPORT_FOLDER": "export",
    "FIGURE_FOLDER": "figures",
    "TRAPZ_DATA_FOLDERS": [],
    "ARIT_DATA_FOLDERS": [],
    "COLOURS": ["green", "magenta", "cyan", "orange"],
    "CHANGE_NAME": True,
    "OWN_AXIS": False,
    "X_AXIS_FILE": "X.txt",
    "UPDATE_FILES": True,
    **DEFAULT_ANALYSIS_PARAMETERS,
})

SETUP = nnmf.select_analysis_setup(
    Path.cwd(), None, COMPONENT_NAMES, DEFAULT_ANALYSIS_PARAMETERS
)
DATA_FOLDER = SETUP["DATA_FOLDER"]
REFERENCE_FILE = SETUP["REFERENCE_FILE"]
LIST_OF_DATA_FOLDERS = SETUP["LIST_OF_DATA_FOLDERS"]
ANALYSIS_PARAMETERS = SETUP["parameters"]
globals().update(ANALYSIS_PARAMETERS)
TRAPZ_DATA_FOLDERS = []
ARIT_DATA_FOLDERS = LIST_OF_DATA_FOLDERS.copy()

nnmf.configure_analysis({
    "DATA_FOLDER": DATA_FOLDER,
    "REFERENCE_FILE": REFERENCE_FILE,
    "EXPORT_FOLDER": "export",
    "FIGURE_FOLDER": "figures",
    "LIST_OF_DATA_FOLDERS": LIST_OF_DATA_FOLDERS,
    "TRAPZ_DATA_FOLDERS": TRAPZ_DATA_FOLDERS,
    "ARIT_DATA_FOLDERS": ARIT_DATA_FOLDERS,
    "COMPONENT_NAMES": COMPONENT_NAMES,
    **ANALYSIS_PARAMETERS,
    "COLOURS": ["green", "magenta", "cyan", "orange"],
    "CHANGE_NAME": True,
    "OWN_AXIS": False,
    "X_AXIS_FILE": "X.txt",
    "UPDATE_FILES": True,
})
EXPORT_DIR = DATA_FOLDER / "export"
FIGURE_DIR = EXPORT_DIR / "figures"
print(f"Working folder: {DATA_FOLDER}")
print(f"Reference workbook: {REFERENCE_FILE}")
print(f"Selected data folders: {LIST_OF_DATA_FOLDERS}")
print(f"Fit components: {COMPONENTS}")
print(f"Reference-initialized components: {INIT_COMP}")


## Setup and reference spectra

The setup window selects the data folder, included subfolders, fit settings, and (optionally) a reference workbook. `INIT_COMP` is the number of components seeded from the workbook: set it to `0` to skip reference loading entirely and use random initialization. When it is greater than `0`, the workbook seeds that many components for one replicate; the other NNMF replicates start randomly. NNMF is free to change the shapes while fitting.


In [ ]:
print(f"Data folder: {DATA_FOLDER}")
print(f"Reference workbook: {REFERENCE_FILE}")
print(f"Data folders: {', '.join(LIST_OF_DATA_FOLDERS)}")
print(f"Baseline band: {BASEMEAN_LOWER}-{BASEMEAN_UPPER} nm")
print(f"Fit range: >{THRESHOLD_LOWER} to {THRESHOLD_UPPER} nm")


## 1. Prepare corrected originals and arithmetic folder averages

Each original has its own baseline subtracted. Spectra with differing wavelength grids are aligned for averaging. The averages are kept for inspection and export; the originals remain separate and are the NNMF inputs.


In [ ]:
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
average_files, individual_files, sample_folders, filenames = nnmf.prepare_analysis_spectra(EXPORT_DIR)
average_wavelength, average_spectra, average_names = nnmf.assemble_tables(
    average_files, EXPORT_DIR / "folder_averages"
)
individual_wavelength, spectra, _ = nnmf.assemble_tables(individual_files, EXPORT_DIR)
print(f"Prepared {len(average_names)} folder averages and {len(filenames)} original spectra.")


## 2. Normalize and crop for NNMF

For **each original spectrum separately**, subtract its baseline, divide by its baseline-corrected intensity at the wavelength nearest NV0, crop to `> THRESHOLD_LOWER` and `<= THRESHOLD_UPPER`, then divide by its own trapezoidal area. Folder averages are processed for inspection but are not fitted.


In [ ]:
average_fit_wavelength, average_normalized = nnmf.normalize_spectra(
    average_wavelength, average_spectra, average_names
)
fit_wavelength, normalized = nnmf.normalize_spectra(
    individual_wavelength, spectra, filenames
)
print(f"NNMF input matrix: {normalized.shape} (wavelengths, original spectra)")


### Inspect processed spectra and small negative values

Values from `-NNMF_NEGATIVE_TOLERANCE` through zero are treated as numerical noise and set to zero for NNMF. More-negative values stop fitting and identify the affected spectrum.


In [ ]:
nnmf.plot_processed_spectra(
    average_fit_wavelength, average_normalized, average_names,
    fit_wavelength, normalized, filenames, sample_folders, FIGURE_DIR
)
negative_counts = np.sum(normalized < 0, axis=0)
for sample_index in np.flatnonzero(negative_counts):
    wavelength_index = np.argmin(normalized[:, sample_index])
    print(filenames[sample_index],
          f"negative points: {negative_counts[sample_index]}",
          f"minimum: {normalized[wavelength_index, sample_index]:.6g}",
          f"at {fit_wavelength[wavelength_index]:.2f} nm")
if SHOW_COMPARED:
    plt.show()


## 3. Jointly fit all original spectra

One NNMF fit receives the whole wavelength-by-spectrum matrix. It learns shared component shapes and separate weights for each original. The first replicate uses workbook initial shapes when enabled; the other replicates start randomly. The lowest-reconstruction-error replicate is retained.


In [ ]:
initial_w = nnmf.load_reference_initialization(fit_wavelength) if INIT_COMP > 0 else None
approximations, weighting = nnmf.fit_joint_spectra(normalized, initial_w, filenames)

final_range = (fit_wavelength >= THRESHOLD_LOWER2) & (fit_wavelength <= THRESHOLD_UPPER2)
if not np.any(final_range):
    raise ValueError("The output range does not overlap the fit range")
wavelength = fit_wavelength[final_range]
measured = normalized[final_range]
shared_shapes = approximations[final_range]
components = shared_shapes[:, None, :] * weighting.T[None, :, :]
fitted = np.sum(components, axis=2)
print(f"Joint fit completed: {len(filenames)} spectra")
print(f"Shared shapes: {approximations.shape}; per-spectrum weights: {weighting.shape}")


## 4. Export fitted results and fit quality

`spect.xlsx` contains measurements, weighted component curves, and fitted spectra. `ratios.xlsx` contains component area percentages and fit-quality measures for each original spectrum, plus an overall mean row.


In [ ]:
ratios_df = nnmf.export_fit_results(
    EXPORT_DIR, wavelength, measured, components, fitted, filenames, sample_folders
)
display(ratios_df.round(4))
print(f"Saved results to {EXPORT_DIR}")


In [ ]:
component_columns = [f"{name} [%]" for name in COMPONENT_NAMES[:COMPONENTS]]
plot_ratios = ratios_df.set_index("Filename")[component_columns]
axis = plot_ratios.plot(
    kind="barh", stacked=True,
    figsize=(9, max(3, 0.4 * len(plot_ratios))),
    color=["green", "magenta", "cyan", "orange"][:COMPONENTS],
)
axis.set_xlabel("Contribution (%)")
axis.set_ylabel("Spectrum")
axis.set_title("Joint NNMF component contributions")
axis.set_xlim(0, 100)
axis.grid(axis="x", alpha=0.25)
plt.tight_layout()
if SHOW_COMPARED:
    plt.show()
else:
    plt.close(axis.figure)


## 5. Plot shared component shapes and individual fits

The shared-shape plot shows the learned shapes normalized to unit area for comparison. Each individual plot shows the measured spectrum, the fit, and its weighted contributions from the shared shapes.


In [ ]:
nnmf.plot_shared_component_shapes(wavelength, shared_shapes, FIGURE_DIR)
nnmf.plot_fit_overviews(wavelength, fitted, filenames, FIGURE_DIR)
nnmf.plot_individual_fits(
    wavelength=wavelength,
    measured=measured,
    fitted=fitted,
    components=components,
    filenames=filenames,
    figure_dir=FIGURE_DIR,
)
print(f"Plots saved to {FIGURE_DIR}")
if SHOW_COMPARED:
    plt.show()
